# Stage 4.5 v4 — TrackNet ball detector (Colab GPU training)

Temporal ball detector trained on the prepared 720p frame caches.
**Runtime → Change runtime type → GPU (T4 or A100).**

## Split (the 5 labeled clips)
- **Train:** pb_3min, pb_4min, pb_5min — same court, different players
- **Val / early-stop:** pb_2min — same court, held out
- **Test (generalization):** pb_3min_court2 — DIFFERENT court, never used for
  training or checkpoint selection → an honest cross-court number

## Upload to Google Drive (e.g. `MyDrive/pb_v4/`)
```
pb_v4/
  repo/                         <- the repo (stages/ package)
  data/pb_2min/  v4_manifest.json + frames_720/
  data/pb_3min/  ...
  data/pb_4min/  ...
  data/pb_5min/  ...
  data/pb_3min_court2/  ...
```
Only `v4_manifest.json` + `frames_720/` per clip — NOT the 4K videos.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/pb_v4')   # <-- adjust if needed
REPO = ROOT / 'repo'
DATA = ROOT / 'data'
sys.path.insert(0, str(REPO))
print('repo:', REPO.exists(), '| data:', DATA.exists())

In [ ]:
try:
    import cv2  # noqa
except Exception:
    !pip -q install opencv-python-headless

In [ ]:
import logging; logging.basicConfig(level=logging.INFO)
from stages.finetune_ball_model.train_v4 import TrainConfig
from stages.finetune_ball_model._v4_cache import train_from_manifests

train_folders = [DATA/'pb_3min', DATA/'pb_4min', DATA/'pb_5min']
val_folder    = DATA/'pb_2min'
test_folder   = DATA/'pb_3min_court2'   # different court = generalization test
for f in train_folders + [val_folder, test_folder]:
    assert (f/'v4_manifest.json').exists(), f'missing manifest: {f}'
print('OK — all manifests present')

In [ ]:
cfg = TrainConfig(
    epochs=40,
    batch_size=8,        # drop to 4 on T4 if OOM; raise on A100
    lr=1e-3,
    out_path=str(ROOT/'ball_model_v4.pt'),   # saved to Drive (persists)
)
result = train_from_manifests(train_folders, val_folder, cfg,
                              test_folder=test_folder, num_workers=4)
print('best VAL recall (same court):', result['best_val_recall'])
print('TEST recall (different court, generalization):', result['test_recall'])

## Reading the result
- **val recall** (same court, new players): should be high — the easy case.
- **test recall** (different court): the number that matters. Acceptance bar
  **≥ 0.80**.
  - **Both high** → the detector generalizes across courts. Done.
  - **Val high, test low** → it overfit to the training court. Fix: add some
    pb_3min_court2 labels into TRAINING (move it to train_folders, label a
    second different court as the new test) — the per-venue fine-tune loop.
  - **Both low** → raise input res to 1080p (set PROC_H,PROC_W=1088,1920 in
    `_v4_data.py`, re-run prepare_v4, retrain).
- `ball_model_v4.pt` + `validation_report.json` are saved to `MyDrive/pb_v4/`.
  Download the .pt to `data/models/` locally and report both recalls.

In [ ]:
import json
rep = json.load(open(ROOT/'validation_report.json'))
print('val clip:', rep['val_clip'], '-> best val recall', rep['best_val_recall'])
print('test clip:', rep['test_clip'], '-> test recall', rep['test_recall_at_best_val'])
for h in rep['history'][-6:]:
    print(h)